# Spot RL Training — Kaggle
Train DQN agent for AWS Spot Instance optimization.


In [ ]:
# Step 1: Install dependencies
!pip install gymnasium torch numpy pandas pyyaml --quiet

In [ ]:
# Step 2: Extract project code from dataset
import zipfile, os, sys

# Kaggle dataset path (sau khi upload spot_rl_kaggle.zip)
ZIP_PATH = '/kaggle/input/spot-rl-code/spot_rl_kaggle.zip'

os.makedirs('/kaggle/working/project', exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall('/kaggle/working/project')

sys.path.insert(0, '/kaggle/working/project')
print('Extracted files:')
for root, dirs, files in os.walk('/kaggle/working/project'):
    for f in files:
        print(' ', os.path.join(root, f).replace('/kaggle/working/project/', ''))

In [ ]:
# Step 3: Verify imports
import numpy as np
import torch
import yaml
import pickle

from envs.spot_orchestrator_env import SpotOrchestratorEnv
from envs.instance_catalog import STATE_DIM, N_ACTIONS, OP_NAMES
from agents.dqn_agent import DQNAgent

print(f'PyTorch: {torch.__version__}')
print(f'State dim: {STATE_DIM}, Action dim: {N_ACTIONS}')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

In [ ]:
# Step 4: Create environment
DATA_PATH = '/kaggle/working/project/data/processed/multipool_stable.csv'

env = SpotOrchestratorEnv(
    data_path=DATA_PATH,
    max_steps=168,
    sla_threshold=0.95,
    workload_config={
        'base_arrival_rate': 15.0,
        'peak_multiplier': 2.0,
        'avg_job_duration': 3,
    },
    seed=42,
)
obs, info = env.reset()
print(f'Env OK — obs shape: {obs.shape}')

In [ ]:
# Step 5: Create agent
agent = DQNAgent(
    state_dim=STATE_DIM,
    action_dim=N_ACTIONS,
    learning_rate=1e-4,
    gamma=0.99,
    epsilon_start=1.0,
    epsilon_end=0.02,
    epsilon_decay=80000,
    batch_size=128,
    buffer_size=200000,
    target_update_freq=1000,
)
print(f'Agent device: {agent.device}')
total_params = sum(p.numel() for p in agent.q_network.parameters())
print(f'Total parameters: {total_params:,}')

In [ ]:
# Step 6: Training loop with early stopping
import time

NUM_EPISODES = 4000
PATIENCE     = 500   # early stopping
MIN_EPISODES = 1000
PRINT_EVERY  = 100
SAVE_DIR     = '/kaggle/working/models'
os.makedirs(SAVE_DIR, exist_ok=True)

rewards_history, loss_history = [], []
cost_history, sla_history, eps_history = [], [], []
best_avg_reward = -float('inf')
no_improve_count = 0
start_time = time.time()

for episode in range(NUM_EPISODES):
    obs, info = env.reset()
    ep_reward, ep_losses = 0.0, []

    for step in range(168):
        action = agent.select_action(obs, training=True)
        next_obs, reward, terminated, truncated, info = env.step(action)
        agent.store_transition(obs, action, reward, next_obs, terminated or truncated)
        loss = agent.train_step()
        if loss is not None:
            ep_losses.append(loss)
        ep_reward += reward
        obs = next_obs
        if terminated or truncated:
            break

    rewards_history.append(ep_reward)
    loss_history.append(np.mean(ep_losses) if ep_losses else 0.0)
    eps_history.append(agent.epsilon)
    cost_history.append(info.get('cost', 0))
    sla_history.append(info.get('sla_compliance', 0))

    # Track best & early stopping
    if episode >= 100:
        avg_r = np.mean(rewards_history[-100:])
        if avg_r > best_avg_reward + 1.0:
            best_avg_reward = avg_r
            no_improve_count = 0
            agent.save(f'{SAVE_DIR}/best_model.pth')
        elif episode >= MIN_EPISODES:
            no_improve_count += 1

    if (episode + 1) % PRINT_EVERY == 0:
        avg_r = np.mean(rewards_history[-100:])
        avg_c = np.mean(cost_history[-100:])
        avg_s = np.mean(sla_history[-100:])
        elapsed = (time.time() - start_time) / 60
        print(f'Ep {episode+1:5d}/{NUM_EPISODES} | '
              f'Reward: {avg_r:+7.1f} | '
              f'Cost: ${avg_c:6.1f} | '
              f'SLA: {avg_s:.1%} | '
              f'Eps: {agent.epsilon:.4f} | '
              f'No-improve: {no_improve_count}/{PATIENCE} | '
              f'Time: {elapsed:.1f}m')

    if episode >= MIN_EPISODES and no_improve_count >= PATIENCE:
        print(f'\nEarly stopping at ep {episode+1}. Best avg: {best_avg_reward:+.1f}')
        break

# Save final model & history
agent.save(f'{SAVE_DIR}/final_model.pth')
with open(f'{SAVE_DIR}/history.pkl', 'wb') as f:
    pickle.dump({
        'rewards': rewards_history, 'loss': loss_history,
        'epsilon': eps_history, 'cost': cost_history, 'sla': sla_history,
    }, f)
print(f'\nDone! Models saved to {SAVE_DIR}')
print(f'Best avg reward: {best_avg_reward:+.1f}')

In [ ]:
# Step 7: Plot training curves
import matplotlib.pyplot as plt
import pandas as pd

def smooth(data, w=50):
    return pd.Series(data).rolling(w, min_periods=1).mean().values

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Training Results — Spot RL v6', fontsize=13)
eps = range(1, len(rewards_history)+1)

axes[0,0].plot(eps, rewards_history, alpha=0.2, color='steelblue')
axes[0,0].plot(eps, smooth(rewards_history), color='steelblue', lw=2)
axes[0,0].set_title('Episode Reward'); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(eps, cost_history, alpha=0.2, color='coral')
axes[0,1].plot(eps, smooth(cost_history), color='coral', lw=2)
axes[0,1].set_title('Cost ($)'); axes[0,1].grid(alpha=0.3)

axes[1,0].plot(eps, sla_history, alpha=0.2, color='green')
axes[1,0].plot(eps, smooth(sla_history), color='green', lw=2)
axes[1,0].axhline(0.95, color='red', linestyle='--', alpha=0.5)
axes[1,0].set_title('SLA'); axes[1,0].grid(alpha=0.3)

axes[1,1].plot(eps, eps_history, color='orange', lw=2)
axes[1,1].set_title('Epsilon'); axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/training_curves.png', dpi=100)
plt.show()
print('Plot saved.')

In [ ]:
# Step 8: Download results
# Kaggle tu dong luu /kaggle/working/ — download tu Output tab
print('Files in output:')
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(f'{SAVE_DIR}/{f}') / 1024
    print(f'  {f}: {size:.0f} KB')
print('\nDownload best_model.pth and history.pkl from Output tab on the right.')